In [ ]:
import pandas as pd
import warnings
from model import LinearRegression, ElasticNet, NN, RandomForest, K_Means_NN, XGBoost, CNN, Encoder
from utils import LoadData
from rolling_train_test import RollingTrainTest
warnings.simplefilter(action='ignore', category=FutureWarning)


In [6]:
import time
# 获取当前计算机时间
current_time = time.strftime("%Y-%m-%d %H:%M:%S", time.localtime())
print("当前时间:", current_time)

当前时间: 2025-09-03 21:58:42


In [ ]:
# encoder
'''close; clean; mse'''
factor = pd.read_csv('../CSV/factor_ma12.csv')
input_dim = factor.shape[1] - 2
# print(f"Input dimension: {input_dim}")

label = pd.read_csv('../CSV/label_cleaned.csv')
# label.describe()
target = 3

# load data and model
Data = LoadData(factor, label, batch_size=32, num_workers=0, shuffle=True)
model_list = [
    Encoder(input_dim, target=target, nlayer=1, flayer=3, d_model=32, nhead=1, dropout=0.1, alpha=0.8, l1_ratio=0.5, model_name="Encoder")
]

# rolling test
count = 0
for model in model_list:
    RTT = RollingTrainTest(model, Data, train_size=0.5, test_size=0.1, epochs=50, patience=5, criterion=None, count=count)
    RTT.info(
        predictability_name = '[--encoder--|--mse--|--close--|epochs=50, patience=5]'
        )
    RTT.run()
    RTT.backtest(trade_mode=1)
    count += 1
    print(f"Model {model.model_name} backtest completed...")
    print("=" * 50)